In [1]:
import pandas as pd

In [2]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t", parse_dates=["time_stamp"])
tdf = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_A" のテストデータだけを抽出
tdf_A = tdf[tdf["user_id"].str.endswith("_A")]

In [4]:
# 行動に対応する重みを定義
event_weights = {
    3: 3,  # 購入
    2: 2,  # 広告クリック
    1: 1,  # 詳細ページの閲覧
    0: 0 # カートに入れる
}

In [5]:
# スコア計算
def calculate_score(row):
    if row["event_type"] == 3 and row["ad"] != 1:
        return 0  # 購入は広告クリック(ad==1)時のみ有効
    return event_weights.get(row["event_type"], 0)

df["score"] = df.apply(calculate_score, axis=1)

In [6]:
# ユーザーごとの時系列特徴量の追加
df["latest_action_time"] = df.groupby("user_id")["time_stamp"].transform("max")
df["time_since_last_action"] = (df["latest_action_time"] - df["time_stamp"]).dt.total_seconds()
df["hour_of_day"] = df["time_stamp"].dt.hour
df["day_of_week"] = df["time_stamp"].dt.dayofweek

In [7]:
# 過去7日間の行動回数

# 各ユーザーのデータを時系列順にソート
df_sorted = df.sort_values(["user_id", "time_stamp"])

# ローリングカウントを計算
rolling_counts = (
    df_sorted.set_index("time_stamp")
    .groupby("user_id")["event_type"]
    .rolling("7d")
    .count()
    .reset_index()
    .rename(columns={"event_type": "actions_last_7d"})
)

# `df` にマージ
df = df.merge(rolling_counts, on=["user_id", "time_stamp"], how="left")

In [8]:
# 夜間（0-6時）の行動割合
df["night_activity"] = (df["hour_of_day"] < 6).astype(int)
df["night_activity_ratio"] = df.groupby("user_id")["night_activity"].transform("mean")

In [9]:
# 週末（土日）の行動割合
df["weekend_activity"] = (df["day_of_week"] >= 5).astype(int)
df["weekend_activity_ratio"] = df.groupby("user_id")["weekend_activity"].transform("mean")

In [10]:
# ユーザー×商品ごとの行動履歴
df["user_product_interaction"] = df.groupby(["user_id", "product_id"])["event_type"].transform("count")
df["user_last_view_time"] = df.groupby(["user_id", "product_id"])["time_stamp"].transform("max")
df["user_time_since_last_view"] = (df["latest_action_time"] - df["user_last_view_time"]).dt.total_seconds()

In [11]:
# 夜間行動割合を加味したスコア
df["adjusted_night_score"] = df["score"] * (1 + df["night_activity_ratio"])

In [12]:
# 週末行動割合を加味したスコア
df["adjusted_weekend_score"] = df["score"] * (1 + df["weekend_activity_ratio"])

In [13]:
# 最終スコア（夜間・週末を考慮）
df["final_score"] = (df["adjusted_night_score"] + df["adjusted_weekend_score"]) / 2

In [14]:
# ユーザー×商品ごとの行動回数を考慮
df["adjusted_interaction_score"] = df["score"] * (1 + df["user_product_interaction"] / df["user_product_interaction"].max())

In [15]:
# 最終スコアに加味
df["final_score"] = (df["final_score"] + df["adjusted_interaction_score"]) / 2

In [16]:
# ユーザー×商品ごとのスコア集計
user_product_score = df.groupby(["user_id", "product_id"], as_index=False)["final_score"].sum()

In [17]:
# ユーザーごとにスコア降順、同スコアなら product_id 昇順で順位付け
user_product_score.sort_values(by=["user_id", "final_score", "product_id"], ascending=[True, False, True], inplace=True)
user_product_score["rank"] = user_product_score.groupby("user_id")["final_score"].rank(method="first", ascending=False).astype(int)

In [18]:
# 特徴量と結合
features = df[["user_id", "product_id", "time_since_last_action", "hour_of_day", "day_of_week", 
               "actions_last_7d", "night_activity_ratio", "weekend_activity_ratio", 
               "user_product_interaction", "user_time_since_last_view"]].drop_duplicates()
user_product_score = user_product_score.merge(features, on=["user_id", "product_id"], how="left")

In [19]:
# 必要なカラムだけを残して重複を削除
user_product_score = user_product_score[["user_id", "product_id", "rank"]].drop_duplicates()

In [20]:
#前処理結果をcsvへ
user_product_score.to_csv('./train/train_A.csv', index=False)
tdf_A.to_csv('./test/test_A.csv', index=False)